# Logistic Regression Baseline Model

The aim here is to use the training and testing split for the logistic regression baseline model. Performance is evaluated on the 2025 test set across key classification metrics: Accuracy, Precision, Recall, F1 Score, ROC-AUC, and Confusion Matrices. A Majority-Class Dummy Classifier is trained as a zero-intelligence reference baseline alongside a Logistic Regression Model. The majority-class dummy classifier serves as an essential zero-intelligence baseline to prove that a machine learning model is genuinely learning useful patterns rather than simply exploiting class imbalance.

In [7]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

categorical_cols = X_train.select_dtypes(
    include=["object", "str", "bool", "category"]
).columns.tolist()
numeric_cols = X_train.select_dtypes(
    exclude=["object", "str", "bool", "category"]
).columns.tolist()

num_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numeric_cols),
        ("cat", cat_transformer, categorical_cols),
    ]
)

dummy_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

dummy_pipeline.fit(X_train, y_train)
y_pred_dummy = dummy_pipeline.predict(X_test)
y_proba_dummy = dummy_pipeline.predict_proba(X_test)[:, 1]

lr_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("=== DUMMY CLASSIFIER METRICS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dummy):.4f}")
print(
    f"Precision: {precision_score(y_test, y_pred_dummy, zero_division=0):.4f}"
)
print(f"Recall: {recall_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_dummy):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dummy))

print("\n=== LOGISTIC REGRESSION METRICS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

=== DUMMY CLASSIFIER METRICS ===
Accuracy: 0.5629
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
ROC-AUC: 0.5000
Confusion Matrix:
 [[483   0]
 [375   0]]

=== LOGISTIC REGRESSION METRICS ===
Accuracy: 0.8147
Precision: 0.7857
Recall: 0.7920
F1 Score: 0.7888
ROC-AUC: 0.8932
Confusion Matrix:
 [[402  81]
 [ 78 297]]

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.83      0.83       483
           1       0.79      0.79      0.79       375

    accuracy                           0.81       858
   macro avg       0.81      0.81      0.81       858
weighted avg       0.81      0.81      0.81       858



The Dummy Classifier achieved an accuracy of 56.29% by predicting the majority class for every observation, resulting in an ROC-AUC of 0.50. In comparison, the Logistic Regression model achieved 81.47% accuracy, with a precision of 78.57%, recall of 79.20%, and F1-score of 78.88%. The model also achieved an ROC-AUC of 0.8932, indicating strong ability to distinguish between the two classes. The confusion matrix shows that the model correctly classified 699 of the 858 test observations. Overall, the Logistic Regression model substantially outperformed the baseline classifier and appears to have learned meaningful relationships between the predictor variables and the target.